# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

## 0. 제출 정보

- 이름: Jongeun Kim
- GitHub ID: risa0122
- 작성일: 2026-10-14
- 최종 제출 URL: https://github.com/risa0122/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb


## 1. 전처리 전 상태 기록

- 각 데이터 shape: customers (152, 6), products (102, 4), orders (302, 5), order_items (765, 5).
- 결측치: age 4건, city 2건으로 총 6셀이다.
- 전체 중복: 네 파일에 각각 2행씩 있다.
- 주요 ID 중복: 네 파일의 기본키에 각각 2건씩 있다.
- 타입 문제 후보: age의 `unknown`, 쉼표 또는 문자가 포함된 금액·수량, 잘못된 날짜가 있다.


In [1]:
from pathlib import Path
import sys
import hashlib
import urllib.request
import pandas as pd
import numpy as np
from IPython.display import display

pd.options.future.infer_string = False
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 20)

CURRENT_DIR = Path.cwd().resolve()
if CURRENT_DIR.name == 'chapter05':
    CHAPTER_DIR = CURRENT_DIR
else:
    CHAPTER_DIR = CURRENT_DIR / 'chapter05'
CHAPTER_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_COMMIT = 'b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc'
SOURCE_BASE = f'https://raw.githubusercontent.com/GilbertMoon/llm-data-analysis-course/{SOURCE_COMMIT}/'
RAW_HASHES = {'customers.csv': 'ef4e554eaae1bb0cb7605aa9e4503aad1e68bbf1eae9093def5c2d04bd26b8cc', 'order_items.csv': 'e59b236328b6b094fdcbc0142af35b7ccb54e4246e1a57180c39997582303e5f', 'orders.csv': 'bb32cb334b9e4920f68e6ef55df17d414bdc51f7c28a58fa3bf1bfb6b2aa5fa1', 'products.csv': 'cad0a825a29078d9451aa74fdfa9e61d474f442d2e377839eca24e9595debbf0'}
SOURCE_HASHES = {'src/data_loader.py': '8c4d1a13861cd08e5a1a3cd60a891d8211ea0e912fac48a0befc0b22e7b42828', 'src/preprocessing.py': '4695e23c80173651d9c1acc62cf6758bc827425f24d7f30b107fa6d3384afcde', 'scripts/preprocess_data.py': '3b4f7db58c19529326fc7466c8c726575d21b8e7d81bfe705c8acb9bdc23fbfd'}
PROJECT_ROOT = next((p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
                     if (p / 'practice/chapter05/data/raw/customers.csv').is_file()
                     and (p / 'src/preprocessing.py').is_file()), None)

# 저장소 밖에서도 동일 버전의 공식 자료를 사용한다.
if PROJECT_ROOT is None:
    PROJECT_ROOT = CHAPTER_DIR / '.official_source'
    downloads = {f'practice/chapter05/data/raw/{name}': digest
                 for name, digest in RAW_HASHES.items()}
    downloads.update(SOURCE_HASHES)
    for rel, expected in downloads.items():
        target = PROJECT_ROOT / rel
        if not target.exists():
            payload = urllib.request.urlopen(SOURCE_BASE + rel, timeout=30).read()
            assert hashlib.sha256(payload).hexdigest() == expected, rel
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)

RAW_DIR = PROJECT_ROOT / 'practice/chapter05/data/raw'
PROCESSED_DIR = CHAPTER_DIR / 'data/processed'
REPORT_DIR = CHAPTER_DIR / 'reports'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

raw_hashes_before = {name: hashlib.sha256((RAW_DIR / name).read_bytes()).hexdigest()
                     for name in RAW_HASHES}
assert raw_hashes_before == RAW_HASHES, '전용 원본이 변경되었는지 확인해야 한다.'
for rel, expected in SOURCE_HASHES.items():
    assert hashlib.sha256((PROJECT_ROOT / rel).read_bytes()).hexdigest() == expected, rel
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python:', sys.version.split()[0], '/ pandas:', pd.__version__)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)
print('공식 전용 Raw 4개 체크섬 일치:', raw_hashes_before == RAW_HASHES)

Python: 3.12.14 / pandas: 3.0.5
원본 데이터 폴더: /Users/jongeun/llm-data-analysis-course/practice/chapter05/data/raw
전처리 데이터 폴더: /Users/jongeun/llm-data-analysis-course/chapter05/data/processed
보고서 폴더: /Users/jongeun/llm-data-analysis-course/chapter05/reports
공식 전용 Raw 4개 체크섬 일치: True


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')

원본 데이터 불러오기 완료


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes

,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()

[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')

전처리용 복사본 생성 완료


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())


[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)

,missing_count,missing_ratio
age,4,2.63
city,2,1.32
customer_id,0,0.00
name,0,0.00
gender,0,0.00
signup_date,0,0.00


In [8]:
# 전처리 전 상태 요약
key_columns = {'customers': 'customer_id', 'products': 'product_id',
               'orders': 'order_id', 'order_items': 'order_item_id'}
before = pd.DataFrame([
    {'데이터': name, '행': len(df), '열': len(df.columns),
     '결측 셀': int(df.isna().sum().sum()),
     '전체 중복': int(df.duplicated().sum()),
     'ID 중복': int(df[key_columns[name]].duplicated().sum())}
    for name, df in raw_data.items()
])
display(before)
print('입력:', RAW_DIR.relative_to(PROJECT_ROOT))
print('전용 원본 체크섬 일치:', raw_hashes_before == RAW_HASHES)

,데이터,행,열,결측 셀,전체 중복,ID 중복
0,customers,152,6,6,2,2
1,products,102,4,0,2,2
2,orders,302,5,0,2,2
3,order_items,765,5,0,2,2


입력: practice/chapter05/data/raw
전용 원본 체크섬 일치: True


![전처리 전 상태](images/step01_before.png)

### 결과 관찰

읽기 직후 결측치는 고객 파일에서만 보이지만, 숫자·날짜처럼 보여도 실제로는 계산할 수 없는 값이 다른 파일에도 있다.

### 나의 해석과 판단

먼저 원본 상태와 중복을 기록하고 타입 변환 실패를 구분한다. 결측 개수만 보고 문제가 없다고 판단하지 않는다.

### 업무·분석적 의미

처리 전 기준을 남겨야 이후 줄어든 행이나 바뀐 값이 의도한 결과인지 설명할 수 있다.

### 한계와 추가 확인 사항

이 데이터는 오류가 포함된 실습용 자료다. 실제 발생 원인이나 운영 시스템의 정정 이력까지 확인할 수는 없다.

## 2. 문자열·타입·결측 처리

- 적용한 문자열 정리: city 5셀, category 3셀, payment_method 5셀의 앞뒤 공백을 제거했다. 빈 문자열이 남는지도 별도로 확인했다.
- 숫자 변환 대상과 실패 건수: age, price, quantity, unit_price를 변환했다. 새 변환 실패는 각각 2·2·5·2건이다.
- 날짜 변환 대상과 실패 건수: signup_date 2건, order_date 3건이 새로 실패했다.
- 결측 처리 기준: age는 원본 결측 4건과 변환 실패 2건을 중앙값 41로 대체했다. city 결측 2건은 Unknown으로 남겼다.


In [9]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)

age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


In [10]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())

customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [11]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')

customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [12]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')

완전 중복 행 제거 완료


In [13]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')

문자열 앞뒤 공백 제거 완료


In [14]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())

city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

In [15]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())

order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

In [16]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()

order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [17]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())

signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


In [18]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())

price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


In [19]:
# 숫자·날짜 변환과 결측 처리 요약
conversion_rows = []
targets = {'customers': ['age', 'signup_date'], 'products': ['price'],
           'orders': ['order_date'], 'order_items': ['quantity', 'unit_price']}
for name, columns in targets.items():
    for column in columns:
        original = raw_data[name][column]
        parsed = (pd.to_datetime(original, errors='coerce')
                  if 'date' in column else to_number(original))
        conversion_rows.append({
            '컬럼': f'{name}.{column}', '원본 결측': int(original.isna().sum()),
            '새 변환 실패': int((original.notna() & parsed.isna()).sum())})
conversion_audit = pd.DataFrame(conversion_rows)
display(conversion_audit)
stage_data = {'customers': customers_clean, 'products': products_clean,
              'orders': orders_clean, 'order_items': order_items_clean}
blank_count = sum(int(df[col].dropna().astype(str).str.strip().eq('').sum())
                  for df in stage_data.values() for col in df.columns)
age_filled = int(pd.to_numeric(customers['age'], errors='coerce').isna().sum())
print('age 대체:', age_filled, '건 / 중앙값:', age_median)
print('city Unknown:', int(customers_clean['city'].eq('Unknown').sum()))
print('처리 후 빈 문자열:', blank_count)

,컬럼,원본 결측,새 변환 실패
0,customers.age,4,2
1,customers.signup_date,0,2
2,products.price,0,2
3,orders.order_date,0,3
4,order_items.quantity,0,5
5,order_items.unit_price,0,2


age 대체: 6 건 / 중앙값: 41.0
city Unknown: 2
처리 후 빈 문자열: 0


![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

원본 결측 6셀 외에 타입 변환으로 16건의 실패가 드러났다. 숫자·날짜 실패 건수는 원본에서 각각 계산한 값이며 같은 행이 둘 이상의 항목에 포함될 수 있다.

### 나의 해석과 판단

나이는 실습 기준에 따라 중앙값으로 대체했다. 평균보다 극단값의 영향을 덜 받지만 분포가 중앙으로 모이는 한계는 남는다.
Unknown은 실제 도시를 추정한 값이 아니라 미확인 범주다. 날짜와 금액은 임의 값으로 채우지 않았다. 결측이 있는 행을 모두 삭제하면 다른 유효한 정보까지 잃으므로 컬럼별로 처리했다.

### 업무·분석적 의미

원래 비어 있던 값과 형식 때문에 읽지 못한 값을 구분하면 수집·입력 단계에서 무엇을 수정해야 하는지 판단할 수 있다.

### 한계와 추가 확인 사항

공식 코드의 나이 중앙값은 완전 중복 제거 전에 계산된다. 이번 실행에서는 41이지만, 실제 분석에서는 중복·범위 검토 후 대체값을 정하는 편이 적절하다.
날짜 실패는 NaT로 남아 있다. 이후 월별 분석에서 이 행들을 누락시키지 않도록 별도 집계해야 한다.

## 3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복: 전체 값이 같은 행을 파일별 2행씩 제거했다. 기본키만 같은 행을 임의로 골라 삭제하지 않았다.
- 표준화한 범주값: 주문 상태의 영문 대소문자·축약·한국어 표기를 completed, cancelled, refunded로 통일했다.
- 허용값 밖 값: 최종 주문 상태에는 없다. completed 180건, cancelled 63건, refunded 57건이다.
- 이상값 후보: 나이 150세, 상품 가격 9,999,999, 수량 100은 추가 확인 대상으로 남겼다. price, quantity, unit_price는 실습 기준에 따라 양수인 행만 남겨 0 이하 값과 숫자 변환 실패값을 제외했다.


In [20]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())

count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [21]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))

price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


In [22]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)

이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


In [23]:
# 나이 범위를 벗어난 후보 확인
age_candidates = customers_clean.loc[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100),
    ['customer_id', 'age']
]
display(age_candidates)
print('후보 수:', len(age_candidates))

,customer_id,age
32,33,150.0


후보 수: 1


In [24]:
# 중복·상태·이상값 후보 요약
processed_data = {'customers': customers_clean, 'products': products_clean,
                  'orders': orders_clean, 'order_items': order_items_clean}
rules_summary = pd.DataFrame([
    {'데이터': name, '전체 중복': int(df.duplicated().sum()),
     'ID 중복': int(df[key_columns[name]].duplicated().sum()),
     '제외 행': len(raw_data[name]) - len(df)}
    for name, df in processed_data.items()
])
display(rules_summary)
print('주문 상태:', orders_clean['order_status'].value_counts().to_dict())
unexpected_status = ~orders_clean['order_status'].isin(['completed', 'cancelled', 'refunded'])
print('허용값 밖 주문 상태:', int(unexpected_status.sum()))
print('나이 18세 미만·100세 초과:', len(age_candidates), '건')
print('추가 확인 후보: 나이 최대', customers_clean['age'].max(),
      '/ 상품 가격 최대', products_clean['price'].max(),
      '/ 수량 최대', order_items_clean['quantity'].max())

,데이터,전체 중복,ID 중복,제외 행
0,customers,0,0,2
1,products,0,0,6
2,orders,0,0,2
3,order_items,0,0,13


주문 상태: {'completed': 180, 'cancelled': 63, 'refunded': 57}
허용값 밖 주문 상태: 0
나이 18세 미만·100세 초과: 1 건
추가 확인 후보: 나이 최대 150.0 / 상품 가격 최대 9999999.0 / 수량 최대 100.0


![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

전처리 후 전체 행 중복과 기본키 중복은 모두 0이다. products는 중복 2행과 가격 조건 4행, order_items는 중복 2행과 수량·단가 조건 11행이 제외됐다.

### 나의 해석과 판단

한 주문에 여러 상품이 있을 수 있으므로 order_items의 order_id 반복은 중복 오류로 보지 않는다.
0원은 증정품, 음수 수량은 반품일 수 있어 실제 업무에서는 같은 기준을 바로 적용하지 않는다. 이번 제외는 공식 실습 범위를 따른 것이다. 큰 값도 실제 고액 상품이나 대량 주문일 수 있으므로 크기만으로 오류라고 단정해 삭제하지 않았다.

### 업무·분석적 의미

상태 표기를 통일해야 같은 상태가 여러 그룹으로 나뉘지 않는다. 제외 사유를 남겨야 특정 거래가 분석에서 사라진 이유도 확인할 수 있다.

### 한계와 추가 확인 사항

양수 조건만으로 큰 양수 이상값을 걸러낼 수 없다. 나이·가격·수량 후보의 실제 의미와 환불·증정 거래 처리 기준은 별도 확인이 필요하다.

## 4. PK/FK와 파생 컬럼 재검증

- 없는 `customer_id`: 주문 2행.
- 없는 `order_id`: 주문 상세 2행.
- 없는 `product_id`: 주문 상세 26행. 모두 서로 다른 ID 26개라는 의미는 아니다.
- 생성한 파생 컬럼: order_month, order_dayofweek, line_total.
- `line_total` 검증: 남긴 주문 상세 752행의 quantity × unit_price와 모두 일치한다.


In [25]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()

,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [26]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())

전처리 후 주문 상세 금액 합계: 252140000.0


In [27]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks

,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


In [28]:
# 상태 전체 기준 월별 고유 주문 수
monthly_orders = orders_clean.groupby('order_month', dropna=False).agg(
    order_count=('order_id', 'nunique')).reset_index()
display(monthly_orders)
print('월을 특정할 수 없는 주문:', orders_clean['order_date'].isna().sum())

,order_month,order_count
0,2025-07,19
1,2025-08,26
2,2025-09,12
3,2025-10,22
4,2025-11,23
5,2025-12,23
6,2026-01,26
7,2026-02,19
8,2026-03,22
9,2026-04,26


월을 특정할 수 없는 주문: 3


In [29]:
# 상태 전체 기준 주문 상세 금액 상위 10행
display(order_items_clean.nlargest(10, 'line_total')[[
    'order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total'
]])

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
10,11,6,97,5.0,187000.0,935000.0


In [30]:
# 관계와 파생 컬럼 재검증
relations = [
    ('orders', 'customer_id', 'customers', 'customer_id'),
    ('order_items', 'order_id', 'orders', 'order_id'),
    ('order_items', 'product_id', 'products', 'product_id')]
relation_audit = pd.DataFrame([
    {'관계': f'{child}.{key} → {parent}.{pk}',
     '처리 전 미매칭 행': int((~raw_data[child][key].isin(raw_data[parent][pk])).sum()),
     '처리 후 미매칭 행': int((~processed_data[child][key].isin(processed_data[parent][pk])).sum())}
    for child, key, parent, pk in relations])
display(relation_audit)
line_matches = order_items_clean['line_total'].eq(
    order_items_clean['quantity'] * order_items_clean['unit_price'])
print('line_total 계산 일치:', bool(line_matches.all()), '/ 합계:', int(order_items_clean['line_total'].sum()))
print('파생 컬럼: order_month, order_dayofweek, line_total')
print('월·요일을 특정할 수 없는 주문:', int(orders_clean['order_date'].isna().sum()))
print('관계 검사 완료. 미매칭은 추가 확인 필요.')

,관계,처리 전 미매칭 행,처리 후 미매칭 행
0,orders.customer_id → customers.customer_id,2,2
1,order_items.order_id → orders.order_id,2,2
2,order_items.product_id → products.product_id,2,26


line_total 계산 일치: True / 합계: 252140000
파생 컬럼: order_month, order_dayofweek, line_total
월·요일을 특정할 수 없는 주문: 3
관계 검사 완료. 미매칭은 추가 확인 필요.


![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

기본키는 결측·중복 없이 유지됐지만 외래키 미매칭은 남았다. 상품을 제외하면 해당 상품을 참조하는 주문 상세가 남을 수 있다.
전체 line_total 합계는 252,140,000이다. 주문 상태 전체의 합계이므로 completed 금액이나 실제 매출로 해석하지 않는다.

### 나의 해석과 판단

전처리 전의 관계 검사만으로는 충분하지 않다. 삭제된 부모 행을 참조하는 자식 행이 새로 생기는지 다시 확인해야 한다.
미매칭을 0으로 맞추기 위해 자동 삭제하지 않았다. 원본에 없던 ID와 상품 제외로 생긴 미매칭을 구분하고, 정정·보류·분석 제외 여부를 결정해야 한다.
현재 결과는 전처리 실습 결과이며, 안전한 병합과 본 분석에 바로 사용 가능한 상태는 아니다.

## 5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 1,321 | 1,298 | 완전 중복 8행, 가격 조건 4행, 수량·단가 조건 11행 제외 |
| 결측 | 6셀 | 5셀 | 원본 컬럼 기준. 나이·도시 보완 후 날짜 변환 실패만 남음 |
| 중복 | 8행 | 0행 | 전체 값이 같은 행만 제거 |
| 변환 실패 | 미확인 | 원본에서 16셀 확인 | 나이 대체·숫자 조건 제외 후 날짜 실패 5셀 잔존 |


In [31]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


In [32]:
# 데이터셋별 컬럼 결측 비율
missing_all = pd.concat([
    missing_summary(df).rename_axis('column').reset_index().assign(dataset=name)
    for name, df in processed_data.items()
], ignore_index=True)[['dataset', 'column', 'missing_count', 'missing_ratio']]
display(missing_all)

,dataset,column,missing_count,missing_ratio
0,customers,signup_date,2,1.33
1,customers,customer_id,0,0.00
2,customers,name,0,0.00
3,customers,gender,0,0.00
4,customers,age,0,0.00
...,...,...,...,...
18,order_items,order_id,0,0.00
19,order_items,product_id,0,0.00
20,order_items,quantity,0,0.00
21,order_items,unit_price,0,0.00


In [33]:
# 전처리 전후 변화 요약
after_summary = pd.DataFrame([
    {'데이터': name, '행 전': len(raw_data[name]), '행 후': len(df),
     '결측 전': int(raw_data[name].isna().sum().sum()),
     '결측 후': int(df.isna().sum().sum()),
     '중복 전': int(raw_data[name].duplicated().sum()),
     '중복 후': int(df.duplicated().sum())}
    for name, df in processed_data.items()
])
display(after_summary)
print('결측 후: 파생 컬럼 포함. 원본 컬럼만의 결측:',
      sum(int(df[raw_data[name].columns].isna().sum().sum())
          for name, df in processed_data.items()))
print('원본 기준 새 변환 실패:', int(conversion_audit['새 변환 실패'].sum()))
print('총 제외 행:', int((after_summary['행 전'] - after_summary['행 후']).sum()))

,데이터,행 전,행 후,결측 전,결측 후,중복 전,중복 후
0,customers,152,150,6,2,2,0
1,products,102,96,0,0,2,0
2,orders,302,300,0,6,2,0
3,order_items,765,752,0,0,2,0


결측 후: 파생 컬럼 포함. 원본 컬럼만의 결측: 5
원본 기준 새 변환 실패: 16
총 제외 행: 23


![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

행 수는 고객 152→150, 상품 102→96, 주문 302→300, 주문 상세 765→752로 총 23행 줄었다. orders에는 두 열, order_items에는 한 열이 추가됐다.
파생 컬럼까지 포함한 결측은 총 8셀이다. 날짜 실패 3건이 order_dayofweek에도 나타나므로 원본 컬럼 기준 5셀과 구분했다.

### 나의 해석과 판단

감소한 행 수는 중복 제거와 양수 조건으로 설명된다. 결측 증가가 항상 품질 악화는 아니다. 잘못된 날짜를 NaT로 표시해 문제가 드러난 결과도 포함된다.
총합·크기만 비교하지 않고 어떤 행과 컬럼에 변화가 생겼는지 함께 봤다.

### 업무·분석적 의미

처리 전후 변화와 남은 문제를 함께 기록하면 다음 분석에서 사용할 범위와 제외 기준을 다시 검토할 수 있다.

### 한계와 추가 확인 사항

order_month에는 날짜 실패가 문자열 `NaT`로 남는다. 월별 분석에서는 실제 월로 취급하지 않는다.
나이 대체로 분포가 달라지고, 상품·주문 상세 제외로 분석 범위도 바뀌므로 처리 전후 결과를 동일 모집단처럼 비교하지 않는다.

## 6. 재현 가능한 전처리 확인

- 생성된 clean CSV: customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv.
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md.
- `python scripts/preprocess_data.py` 재실행 결과: 같은 원본과 src 모듈을 별도 폴더에 복사해 2회 실행했다. pandas 문자열 옵션은 Notebook과 동일하게 적용했으며, CSV 4개와 보고서 내용이 두 실행에서 일치했다.
- 재실행 후 달라진 점: 결과값과 원본 체크섬은 변하지 않았다. Notebook·함수·모듈·스크립트 결과 및 저장 CSV 재로딩 결과가 일치했다.


In [34]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))

[PosixPath('/Users/jongeun/llm-data-analysis-course/chapter05/data/processed/orders_clean.csv'),
 PosixPath('/Users/jongeun/llm-data-analysis-course/chapter05/data/processed/customers_clean.csv'),
 PosixPath('/Users/jongeun/llm-data-analysis-course/chapter05/data/processed/order_items_clean.csv'),
 PosixPath('/Users/jongeun/llm-data-analysis-course/chapter05/data/processed/products_clean.csv')]

In [35]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)

요약 보고서 저장 완료: /Users/jongeun/llm-data-analysis-course/chapter05/reports/ch05_preprocessing_summary.md


In [36]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()

In [37]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)

(150, 6)
(96, 4)
(300, 7)
(752, 6)


In [38]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [39]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)

,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [40]:
module_relationship_checks

,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


In [41]:
# 공식 스크립트 재실행과 저장 결과 대조
import os
import shutil
import subprocess
import tempfile

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

reproduction_rows = []
function_data = {'customers': customers_clean_func, 'products': products_clean_func,
                 'orders': orders_clean_func, 'order_items': order_items_clean_func}
for name, df in processed_data.items():
    pd.testing.assert_frame_equal(df.reset_index(drop=True), function_data[name].reset_index(drop=True))
    pd.testing.assert_frame_equal(df.reset_index(drop=True), module_processed_data[name].reset_index(drop=True))
reproduction_rows.append({'확인': 'Notebook·함수·모듈 결과', '결과': '4개 모두 일치'})

with tempfile.TemporaryDirectory(prefix='ch05-reproduce-') as tmp:
    run_root = Path(tmp)
    (run_root / 'practice/chapter05/data/raw').mkdir(parents=True)
    for filename in RAW_HASHES:
        shutil.copy2(RAW_DIR / filename, run_root / 'practice/chapter05/data/raw' / filename)
    for rel in SOURCE_HASHES:
        target = run_root / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PROJECT_ROOT / rel, target)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(run_root)
    # 공식 object 문자열 처리와 같은 옵션으로 독립 프로세스를 실행한다.
    runner = ("import pandas as pd, runpy; pd.options.future.infer_string=False; "
              "runpy.run_path('scripts/preprocess_data.py', run_name='__main__')")
    runs = []
    for _ in range(2):
        result = subprocess.run([sys.executable, '-c', runner], cwd=run_root,
                                env=env, text=True, capture_output=True, check=True)
        hashes = {p.name: file_hash(p) for p in (run_root / 'data/processed').glob('*_clean.csv')}
        hashes['report'] = file_hash(run_root / 'reports/ch05_preprocessing_summary.md')
        runs.append(hashes)
    assert runs[0] == runs[1]
    assert len(runs[0]) == 5
    reproduction_rows.append({'확인': '스크립트 1·2차 재실행', '결과': 'CSV 4개·보고서 동일'})
    for name in processed_data:
        saved = pd.read_csv(PROCESSED_DIR / f'{name}_clean.csv')
        replay = pd.read_csv(run_root / 'data/processed' / f'{name}_clean.csv')
        pd.testing.assert_frame_equal(saved, replay)
        assert list(saved.columns) == list(processed_data[name].columns)
        assert len(saved) == len(processed_data[name])
        assert saved[key_columns[name]].is_unique
    reproduction_rows.append({'확인': '저장 CSV 재로딩·스크립트 대조', '결과': '4개 모두 일치'})
    replay_log = result.stdout

reload_customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
reload_orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
assert int(pd.to_datetime(reload_customers['signup_date'], errors='coerce').isna().sum()) == 2
assert int(pd.to_datetime(reload_orders['order_date'], errors='coerce').isna().sum()) == 3
raw_preserved = {name: file_hash(RAW_DIR / name) for name in RAW_HASHES} == raw_hashes_before
assert raw_preserved
reproduction_rows.append({'확인': '전용 Raw 원본 보존', '결과': '4개 체크섬 동일'})
reproduction_rows.append({'확인': '날짜 재로딩', '결과': '가입일 2건·주문일 3건 결측 유지'})
reproduction_rows.append({'확인': '남은 품질 문제', '결과': '미매칭 2·2·26행, 날짜 실패 등 추가 확인'})
reproduction = pd.DataFrame(reproduction_rows)

In [42]:
# 재현성 검증 결과
display(reproduction)
print('저장:', PROCESSED_DIR.relative_to(CHAPTER_DIR), '/', report_path.relative_to(CHAPTER_DIR))
print('원본 보존·재실행 검증 완료. 데이터 품질 전체 통과를 의미하지 않음.')

,확인,결과
0,Notebook·함수·모듈 결과,4개 모두 일치
1,스크립트 1·2차 재실행,CSV 4개·보고서 동일
2,저장 CSV 재로딩·스크립트 대조,4개 모두 일치
3,전용 Raw 원본 보존,4개 체크섬 동일
4,날짜 재로딩,가입일 2건·주문일 3건 결측 유지
5,남은 품질 문제,"미매칭 2·2·26행, 날짜 실패 등 추가 확인"


저장: data/processed / reports/ch05_preprocessing_summary.md
원본 보존·재실행 검증 완료. 데이터 품질 전체 통과를 의미하지 않음.


In [43]:
# 스크립트의 실제 실행 로그
print(replay_log)

전처리 완료
입력 Raw 경로: practice/chapter05/data/raw

[전처리 전후 데이터 크기]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             752                  6
     orders       302            5             300                  7
   products       102            4              96                  4

[중복 점검 결과]
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0

[파일 간 관계 점검 결과]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              2
      order_items.order_id exists in orders.order_id              2
order_items.pr

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단

한 번의 실행 성공만으로는 셀 실행 순서나 남아 있는 변수의 영향을 배제할 수 없다.
별도 프로세스에서 원본부터 다시 계산해 같은 결과가 나오는지 확인해야 처리 과정을 재현할 수 있다.
결과 재현과 데이터 품질 통과는 다르다. 같은 결과가 재현돼도 날짜 실패와 미매칭 문제는 남아 있다.

## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본을 보존하고 복사본에서 처리한다. 처리 전후 상태와 원본 체크섬을 남긴다.
2. 값의 의미를 확인하고 처리 기준을 정한다. 원본 결측과 변환 실패, 전체 행 중복과 ID 반복을 구분한다.
3. 전처리 후 PK/FK, 파생값, 저장 파일을 재검증한다. 코드 실행 성공을 품질 통과로 보지 않는다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

미매칭 행을 이유 확인 없이 모두 삭제하는 처리다. 부모 파일에서 제외한 행 때문에 생긴 문제일 수도 있고,
실제 거래가 누락돼 금액과 고객 구성이 달라질 수 있다. 원인을 분리하고 제외 규모를 확인한 뒤 판단해야 한다.

### 다음 EDA에서 특히 주의할 데이터 특성

- 날짜 실패 5건과 order_month의 문자열 NaT를 별도로 관리한다.
- 150세, 큰 상품 가격·수량이 분포와 평균에 미치는 영향을 확인한다.
- 미매칭 2·2·26행의 처리 기준을 먼저 정하고 병합 손실을 확인한다.
- 주문 상태 전체와 completed 범위를 구분한다. line_total 합계를 실제 매출·순매출·이익으로 부르지 않는다.

### 현재 전처리의 한계

공식 실습 기준으로 처리했으며 업무 원장이나 정정 사유는 확인하지 못했다.
나이 중앙값 대체, 양수 조건 제외가 실제 업무에서도 적절한지는 검증되지 않았다.
남은 미매칭·날짜 실패·이상값 후보를 해결하기 전에는 결과를 일반화하지 않는다.

**LLM 제안과 검증**

**제공한 요약과 질문:** 고객 152행, age 원본 결측 4건·숫자 변환 실패 2건, city 결측 2건, 각 파일 전체 중복 2행이다.
전처리 후 미매칭은 2·2·26행이다. 고객 원본 행 대신 이 요약과 컬럼 구조로 결측 처리 후보와 위험한 자동 처리를 검토해 달라고 요청했다.

**Codex 검토:** age의 삭제·평균·중앙값 대체는 각각 표본 손실·극단값 영향·분포 축소의 한계가 있다.
city Unknown은 미확인 범주로 보존하고, 날짜 실패를 임의 날짜로 대체하지 않는 것이 타당하다.
기본키 중복이나 미매칭을 자동 삭제하지 않고, 양수 필터가 변환 실패도 제외한다는 점을 기록해야 한다.

**반영 및 판단:** 공식 실습의 중앙값·Unknown·양수 조건을 적용하되 원본 결측과 변환 실패를 구분해 계산했다.
제외 행 수, 남은 관계 문제, 재실행 결과를 실제 출력과 비교했다. 근거 없는 날짜·ID 대체는 적용하지 않았다.

## 최종 제출 체크

- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.
